# Excel Chat Analyst  +  Build a Coding Agent

**Part A** - upload a spreadsheet, ask questions in plain English, get real tables and charts.
**Part B** - take the same idea and build a general **coding agent**.

## How Part A works

1. The user **uploads a spreadsheet** in the app.
2. `understand_file()` loads it into a pandas `DataFrame` and builds a short **schema**
   (column names, types, a few sample rows). The app says **"ready for chat"**.
3. The user asks a question. The model writes **pandas code** as text - it does no arithmetic itself.
4. We `exec()` that code here against the real `df`.
5. If it errors, send the error back once and let the model fix it.
6. Send the printed output back to the model for a plain-English answer.

**Two things to remember:**
- Your data stays in this runtime - only the schema is sent out.
- pandas computes every number, so the figures are correct, not guessed.

> **Safety note:** step 4 runs model-written code with `exec()`. That is fine for *your own*
> files on a throwaway Colab VM. For untrusted files, run the code in a sandbox instead
> (a subprocess with restricted permissions, or a container) - `run_code` is the one place to
> change.


## A.1  Install

In [ ]:
!pip install -q gradio groq openpyxl

## A.2  Imports

One small change to matplotlib: bigger default figures and a light grid. That's all the styling this notebook does.

In [ ]:
import io, os, re, contextlib, traceback
from uuid import uuid4
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")                 # save charts as PNG, don't try to show them inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

import gradio as gr
from groq import Groq

CHART_DIR = "charts"
os.makedirs(CHART_DIR, exist_ok=True)
print("gradio", gr.__version__, "| pandas", pd.__version__)

## A.3  Connect to Groq

Free key: https://console.groq.com/keys . On Colab, add it as a secret named `GROQ_API_KEY`
(key icon, left sidebar); otherwise the cell asks you to paste it.

In [ ]:
import getpass

key = os.environ.get("GROQ_API_KEY")
if not key:
    try:
        from google.colab import userdata
        key = userdata.get("GROQ_API_KEY")
    except Exception:
        key = None
if not key:
    key = getpass.getpass("Paste your Groq API key (gsk_...): ").strip()
os.environ["GROQ_API_KEY"] = key

client = Groq(api_key=key, max_retries=6)     # max_retries handles the free-tier rate limit

PREFERRED = ["openai/gpt-oss-120b", "qwen/qwen3.8-27b", "openai/gpt-oss-20b", "llama-3.3-70b-versatile"]
available = [m.id for m in client.models.list().data]
MODEL = next((m for m in PREFERRED if m in available),
             next(m for m in available if not any(x in m for x in ("whisper", "guard", "tts"))))
print("Using model:", MODEL)

## A.4  Understand the uploaded file

There is **no sample file** - the user uploads their own spreadsheet in the app. Two functions
do all the "understanding":

- `profile_dataframe(df)` - turns a table of any size into a short text description. **This text
  is the only thing sent to the model.** Text columns with few distinct values are listed in
  full (so filters are exact); columns with many distinct values (names, ids) are withheld.
- `understand_file(path)` - the single entry point the app calls once per upload: open the file,
  load the first sheet, profile it, and return the DataFrame + schema + a one-line summary.

In [ ]:
SAMPLE_ROWS = 3        # set to 0 to send no cell values at all
MAX_ENUMERATE = 25    # text columns with more distinct values than this are not listed

def profile_dataframe(df, sample_rows=SAMPLE_ROWS):
    lines = [f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns", "", "COLUMNS:"]
    for col in df.columns:
        s = df[col]
        bits = [f"dtype={s.dtype}"]
        n_null = int(s.isna().sum())
        if n_null:
            bits.append(f"nulls={n_null:,}")
        n_uniq = int(s.nunique(dropna=True))
        bits.append(f"unique={n_uniq:,}")
        if pd.api.types.is_datetime64_any_dtype(s) and s.notna().any():
            bits.append(f"range={s.min():%Y-%m-%d} to {s.max():%Y-%m-%d}")
        elif pd.api.types.is_numeric_dtype(s) and s.notna().any():
            bits.append(f"min={s.min():,.2f} max={s.max():,.2f} mean={s.mean():,.2f}")
        elif n_uniq > MAX_ENUMERATE:
            bits.append("high-cardinality text (values withheld)")
        else:
            bits.append("values=[" + ", ".join(sorted(map(str, s.dropna().unique()))) + "]")
        lines.append(f"  - {col} | " + " | ".join(bits))
    if sample_rows > 0:
        lines += ["", f"FIRST {sample_rows} ROWS:", df.head(sample_rows).to_string()]
    return "\n".join(lines)

def read_sheet(path, sheet_name):
    df = pd.read_excel(path, sheet_name=sheet_name)
    df.columns = [str(c).strip().replace("\n", " ") for c in df.columns]
    return df

def understand_file(path, sheet=None):
    """Called ONCE when a file is uploaded. Returns (df, schema_text, short_summary)."""
    names = pd.ExcelFile(path).sheet_names
    sheet = sheet or names[0]
    df = read_sheet(path, sheet)
    schema = profile_dataframe(df)
    n_null = int(df.isna().sum().sum())
    extra = f"  (file also has: {', '.join(n for n in names if n != sheet)})" if len(names) > 1 else ""
    summary = (f"Loaded sheet **{sheet}** - {len(df):,} rows x {len(df.columns)} columns, "
               f"{n_null:,} missing values.{extra}\n\n"
               f"Columns: {', '.join(map(str, df.columns))}")
    return df, schema, summary

print("profiler ready")

## A.6  Turn the question into pandas code

The system prompt sets the contract: assign the answer to `result`, `print()` the key numbers,
draw a chart when it helps, never touch files or the network, never call `plt.show()`.

In [ ]:
CODE_SYSTEM_PROMPT = """You are a data analyst who writes short, correct pandas code.

You get the SCHEMA of a DataFrame named `df` (already in memory) and a question. Write code that
answers it when run against the real `df`. You only see the schema, not the full data.

RULES
1. `df`, `pd`, `np`, `plt` already exist. Import nothing.
2. Assign the main answer to a variable `result` (a DataFrame or Series). For a single number:
   result = pd.DataFrame({"Metric": ["Total revenue"], "Value": [1234.0]})
3. print() the key numbers - those printed lines are the only evidence for the written answer.
4. Draw a chart with plt for any trend, comparison, ranking or distribution.
   NEVER call plt.show() or plt.savefig() - just create the figure with: fig, ax = plt.subplots()
5. Never use files, network, os, sys, subprocess, eval, exec, open. Never install anything.
6. Handle nulls explicitly (dropna / fillna) and say what you did.
7. Call pd.to_datetime(df[col]) before any .dt accessor.
8. Use only columns that appear in the schema.

CHART RULES: one y-axis only; sort bars by value; at most 8 series; always a title and axis
labels; don't set explicit colours.

Return ONLY one python code block:
```python
# code
```"""

def extract_code(text):
    m = re.search(r"```(?:python)?\s*\n(.*?)```", text, re.S)
    return (m.group(1) if m else text).strip()

def generate_code(question, schema, prev_code=None, error=None):
    user = f"SCHEMA OF `df`:\n{schema}\n\nQUESTION: {question}"
    if error:
        user += ("\n\n---\nYour previous attempt FAILED. Return the corrected full script.\n\n"
                 f"PREVIOUS CODE:\n{prev_code}\n\nERROR:\n{error}")
    resp = client.chat.completions.create(
        model=MODEL, temperature=0.1, max_tokens=1600,
        messages=[{"role": "system", "content": CODE_SYSTEM_PROMPT},
                  {"role": "user", "content": user}])
    return extract_code(resp.choices[0].message.content)

print("code generator ready")

## A.7  Run the generated code

`run_code` executes the model's code against a **copy** of `df` (so a bad `dropna(inplace=True)`
can't corrupt your data) and captures four things: what was printed, the `result` object, any
charts, and the traceback if it failed.

> For **untrusted** spreadsheets, swap the `exec` line for a sandbox (subprocess with restricted
> permissions, or a container). For your own files on Colab, `exec` is fine.

In [ ]:
def run_code(code, df):
    plt.close("all")
    ns = {"df": df.copy(), "pd": pd, "np": np, "plt": plt}
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, ns)                     # <-- replace with a sandbox for untrusted files
    except Exception:
        plt.close("all")
        return {"ok": False, "stdout": buf.getvalue(),
                "error": traceback.format_exc(limit=2), "result": None, "figures": []}
    figures = []
    for num in plt.get_fignums():
        fig = plt.figure(num)
        if fig.get_axes():
            path = os.path.join(CHART_DIR, f"chart_{uuid4().hex[:8]}.png")
            fig.savefig(path, dpi=120, bbox_inches="tight")
            figures.append(path)
    plt.close("all")
    return {"ok": True, "stdout": buf.getvalue(), "error": None,
            "result": ns.get("result"), "figures": figures}

print("executor ready")

## A.8  Explain the result in plain English

A second, cheap call turns the captured output into 2-4 sentences. Every number it writes must come from the output - it captions numbers, it doesn't compute them.

In [ ]:
ANSWER_SYSTEM_PROMPT = """You explain a data result to a colleague.
You get a question, the code that ran, and its ACTUAL OUTPUT. Every number you write must appear
in that output - never invent or estimate. Write 2-4 short plain-English sentences: the direct
answer with the key figures first, then one line of interpretation if something stands out, and
a short mention of the chart if there is one. No headings, no "based on the output"."""

def summarise(question, code, run):
    ev = []
    if run["stdout"].strip():
        ev.append("PRINTED OUTPUT:\n" + run["stdout"].strip()[:4000])
    res = run["result"]
    if isinstance(res, (pd.DataFrame, pd.Series)):
        frame = res.to_frame() if isinstance(res, pd.Series) else res
        ev.append(f"RESULT TABLE (first 25 of {len(frame):,}):\n" + frame.head(25).to_string())
    if run["figures"]:
        ev.append(f"CHARTS PRODUCED: {len(run['figures'])}")
    if not ev:
        return "The code ran but produced no output. Try rephrasing the question."
    resp = client.chat.completions.create(
        model=MODEL, temperature=0.2, max_tokens=500,
        messages=[{"role": "system", "content": ANSWER_SYSTEM_PROMPT},
                  {"role": "user", "content": f"QUESTION: {question}\n\nCODE:\n{code}\n\n" + "\n\n".join(ev)}])
    return resp.choices[0].message.content.strip()

print("summariser ready")

## A.9  The orchestrator

Generate → screen → run → retry once with the error → explain. The retry passes the **traceback** back, which is usually enough for the model to fix a wrong column name.

In [ ]:
def to_display_frame(obj, max_rows=200):
    if obj is None:
        return None
    if isinstance(obj, pd.Series):
        obj = obj.rename("Value").reset_index()
    if isinstance(obj, pd.DataFrame):
        out = obj.head(max_rows).copy()
        if not isinstance(out.index, pd.RangeIndex):
            out = out.reset_index()
        for c in out.columns:
            if pd.api.types.is_datetime64_any_dtype(out[c]):
                out[c] = out[c].dt.strftime("%Y-%m-%d")
            elif pd.api.types.is_float_dtype(out[c]):
                out[c] = out[c].round(2)
        out.columns = [str(c) for c in out.columns]
        return out
    return pd.DataFrame({"Value": [str(obj)]})

def answer_question(question, df, schema=None, verbose=False):
    schema = schema or profile_dataframe(df)     # reuse the schema built at upload time
    code = generate_code(question, schema)
    run = run_code(code, df)
    attempts = 1
    if not run["ok"]:
        if verbose: print("attempt 1 failed:", run["error"].splitlines()[-1])
        code = generate_code(question, schema, prev_code=code, error=run["error"])
        run = run_code(code, df)
        attempts = 2
    if not run["ok"]:
        return {"answer": "Could not produce a working analysis.\n\n```\n"
                + run["error"].strip()[-800:] + "\n```", "table": None, "figures": [],
                "code": code, "status": f"failed after {attempts} attempts"}
    return {"answer": summarise(question, code, run),
            "table": to_display_frame(run["result"]),
            "figures": run["figures"], "code": code,
            "status": f"answered in {attempts} attempt(s), {len(run['figures'])} chart(s)"}

print("pipeline ready")

## A.10  The Gradio app

The flow: **upload -> `understand_file()` runs once -> "ready for chat" -> chat**.

- `on_upload` calls `understand_file` and stashes the DataFrame **and** the schema in
  `gr.State`, so every later question reuses the same schema (no re-profiling).
- The chat box is disabled until a file is understood.
- The written answer goes in the chat; the table, charts and generated code for the **latest**
  question show underneath.

In [ ]:
SAMPLES = [
    "Give me a high-level summary of this data - what's in it and what stands out?",
    "Which category or group has the highest total, and which has the lowest?",
    "Show a trend over time and name the best and worst periods.",
    "Are any two numeric columns related? Show a scatter plot.",
    "Which rows look like outliers, and why?",
]

def on_upload(path):
    """Runs once when a file is uploaded: understand it, then enable the chat."""
    if not path:
        return None, "Upload an Excel file to begin.", gr.update(interactive=False), []
    try:
        df, schema, summary = understand_file(path)
    except Exception as exc:
        return None, f"Could not read that file:\n```\n{exc}\n```", gr.update(interactive=False), []
    state = {"df": df, "schema": schema}
    return (state,
            "**Ready for chat.**  " + summary,
            gr.update(interactive=True, placeholder="Ask a question about your data..."),
            [])

def on_ask(question, chat_history, state):
    chat_history = chat_history or []
    if state is None:
        chat_history += [{"role": "user", "content": question},
                         {"role": "assistant", "content": "Upload a file first."}]
        return chat_history, "", None, [], ""
    if not question.strip():
        return chat_history, "", None, [], ""

    out = answer_question(question.strip(), state["df"], schema=state["schema"])
    chat_history += [{"role": "user", "content": question},
                     {"role": "assistant", "content": out["answer"] + f"\n\n_{out['status']}_"}]
    return chat_history, "", out["table"], out["figures"], out["code"]

with gr.Blocks(title="Excel Chat Analyst") as demo:
    gr.Markdown("# Excel Chat Analyst\nUpload a spreadsheet, wait for **\"ready for chat\"**, "
                "then ask questions. Only the column names + a few sample rows are sent to the model.")
    state = gr.State(None)

    file_in = gr.File(label="Upload an Excel file (.xlsx / .xls)",
                      file_types=[".xlsx", ".xls"], type="filepath")
    status = gr.Markdown("Upload an Excel file to begin.")

    chat = gr.Chatbot(label="Chat", height=380)
    q = gr.Textbox(label="Message", interactive=False, placeholder="Upload a file first...")
    gr.Examples(SAMPLES, inputs=q, label="Example questions")

    with gr.Accordion("Latest result: table / charts / generated code", open=True):
        table = gr.Dataframe(label="Result table", interactive=False, wrap=True)
        charts = gr.Gallery(label="Charts", columns=2, height=380)
        code_box = gr.Code(language="python", label="Generated pandas code")

    file_in.change(on_upload, file_in, [state, status, q, chat])
    q.submit(on_ask, [q, chat, state], [chat, q, table, charts, code_box])

demo.launch(share=True, debug=True)

---
# Part B - Build a Coding Agent

The Excel analyst above **is a coding agent with training wheels**: it writes code, runs it, reads
the error, tries again. Take away the "it must be about a DataFrame" restriction and you have a
**general coding agent** - the pattern behind tools like Code Interpreter.

The loop:

```
task ->  model writes Python  ->  run it  ->  model sees the output/error
              ^                                         |
              |_______________  not done yet  __________|
                                                        |
                                                     DONE  ->  final answer
```

The one new idea vs Part A: a **persistent namespace**. Variables created in step 1 are still
there in step 2, so the agent can build up a solution across turns, like a REPL.

## B.1  Run one step of code (persistent namespace)

In [ ]:
def run_step(code, ns):
    """Run one code block in the shared namespace `ns`. Return printed output or the error."""
    ns.setdefault("pd", pd); ns.setdefault("np", np); ns.setdefault("plt", plt)
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, ns)                     # same 'trust your own runtime' note as Part A
    except Exception:
        return "ERROR:\n" + traceback.format_exc(limit=2)
    out = buf.getvalue().strip()
    return out if out else "(ran with no output)"

print("run_step ready")

## B.2  The agent loop

`coding_agent(task)` alternates: ask the model for code, run it, feed the result back. It stops
when the model writes `DONE` or after `max_steps`. Every step is printed so you can watch it
work.

In [ ]:
AGENT_SYSTEM_PROMPT = """You are a coding agent. You solve the user's task by writing and
running Python, one step at a time.

Each turn, reply with EITHER:
  - a single ```python code block``` to run next (it shares a namespace with your previous code,
    so variables persist), OR
  - the word DONE on its own line, followed by a short plain-English answer, once the task is
    finished and you have printed the result.

Rules: print() what you want to see - that output is fed back to you. No files, no network, no
os/sys/subprocess/eval/exec/open, no installs. For a chart, create the figure (no plt.show)."""

def coding_agent(task, max_steps=6, verbose=True):
    ns = {}
    messages = [{"role": "system", "content": AGENT_SYSTEM_PROMPT},
                {"role": "user", "content": f"TASK: {task}"}]
    if verbose:
        print("TASK:", task, "\n" + "=" * 70)
    for step in range(1, max_steps + 1):
        reply = client.chat.completions.create(
            model=MODEL, temperature=0.1, max_tokens=1200, messages=messages
        ).choices[0].message.content
        messages.append({"role": "assistant", "content": reply})

        if "DONE" in reply.split("\n")[0] or reply.strip().startswith("DONE"):
            final = reply.split("DONE", 1)[1].strip(" :\n")
            if verbose:
                print(f"\n[step {step}] DONE\n{final}")
            return {"answer": final, "steps": step, "namespace": ns}

        if "```" not in reply:                       # no code and not DONE -> nudge
            messages.append({"role": "user", "content": "Reply with a ```python code block``` to run, or DONE."})
            continue
        code = extract_code(reply)
        result = run_step(code, ns)
        if verbose:
            print(f"\n[step {step}] code:\n{code}\n--- output ---\n{result[:800]}")
        messages.append({"role": "user", "content": f"OUTPUT:\n{result}\n\nNext step, or DONE."})

    return {"answer": "(stopped: hit max_steps)", "steps": max_steps, "namespace": ns}

print("coding_agent ready")

## B.3  Watch it solve a few tasks

In [ ]:
_ = coding_agent("Estimate the value of pi with a Monte-Carlo simulation of 200000 random points. Print the estimate and the error vs math.pi.")

In [ ]:
_ = coding_agent("Make a DataFrame of 12 months of made-up monthly revenue, find the 3 highest months, print them, and draw a bar chart of all 12.")

In [ ]:
_ = coding_agent("A list of 500 integers is called data. Some are None. Compute the mean, ignoring None, without using numpy or pandas. First create the list yourself.")

## B.4  A Gradio front-end for the coding agent

In [ ]:
def agent_ui(task):
    if not task.strip():
        return "Type a task.", ""
    trace_lines = []
    ns = {}
    messages = [{"role": "system", "content": AGENT_SYSTEM_PROMPT},
                {"role": "user", "content": f"TASK: {task}"}]
    for step in range(1, 7):
        reply = client.chat.completions.create(
            model=MODEL, temperature=0.1, max_tokens=1200, messages=messages
        ).choices[0].message.content
        messages.append({"role": "assistant", "content": reply})
        if reply.strip().startswith("DONE") or "DONE" in reply.split("\n")[0]:
            return reply.split("DONE", 1)[1].strip(" :\n"), "\n\n".join(trace_lines)
        if "```" not in reply:
            messages.append({"role": "user", "content": "Reply with a ```python code block``` to run, or DONE."})
            continue
        code = extract_code(reply)
        result = run_step(code, ns)
        trace_lines.append(f"### Step {step}\n```python\n{code}\n```\n**Output**\n```\n{result[:1000]}\n```")
        messages.append({"role": "user", "content": f"OUTPUT:\n{result}\n\nNext step, or DONE."})
    return "(stopped: hit max steps)", "\n\n".join(trace_lines)

with gr.Blocks(title="Coding Agent") as agent_demo:
    gr.Markdown("# Coding Agent\nGive it a task. It writes Python, runs it, reads the output, and iterates.")
    task_in = gr.Textbox(label="Task", lines=2,
                         value="Generate 1000 samples from a normal distribution, then print the mean, std, and a histogram.")
    go = gr.Button("Run", variant="primary")
    ans = gr.Markdown(label="Answer")
    with gr.Accordion("Steps the agent took", open=True):
        tr = gr.Markdown()
    gr.Examples([
        "Find all prime numbers below 100 and print them.",
        "Make fake sales data for 5 products over 6 months and plot each product's trend.",
        "Sort the list [5,2,9,1,7,3] using merge sort you implement yourself. Show it works.",
    ], inputs=task_in)
    go.click(agent_ui, task_in, [ans, tr])

agent_demo.launch(share=True, debug=False)

## Recap

| | Excel Chat Analyst (Part A) | Coding Agent (Part B) |
|---|---|---|
| Scope | one question about one `df` | any Python task |
| Memory | none - each question is fresh | persistent namespace across steps |
| Loop | generate → run → **1 retry** on error → explain | generate → run → observe → repeat until **DONE** |
| Safety | `exec` on a df copy (swap for a sandbox on untrusted files) | `exec` in a persistent namespace |

Both are the same core idea: **let the model write code, run the code yourself, feed back what
happened.** The model never does the computation - it decides *what* to compute.

### Exercises
1. Give the Excel analyst memory: keep the last 2 (question, code) pairs and pass them into
   `generate_code`, so "now split that by segment" works.
2. Wire the two parts together: give the Part B agent a `load_excel(path)` helper and let it
   answer questions about an uploaded spreadsheet across several steps.
3. Make the coding agent stop early if it repeats the same failing code twice.
